# 🚢 Titanic Survival Prediction
### Data Science Project | Machine Learning with Logistic Regression

---

**Objective:** Predict whether a passenger survived the Titanic disaster using machine learning.

**Pipeline:**
1. Data Collection & Loading
2. Exploratory Data Analysis (EDA)
3. Data Cleaning & Preprocessing
4. Feature Engineering
5. Data Visualization
6. Model Building (Logistic Regression)
7. Model Evaluation

---
## Step 1: Import Libraries

In [ ]:
# Core Libraries
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

# Settings
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

print("✅ All libraries imported successfully!")

---
## Step 2: Data Collection — Load the Titanic Dataset

In [ ]:
# Load Titanic dataset directly from seaborn's built-in datasets
df = sns.load_dataset('titanic')

print(f"Dataset Shape: {df.shape}")
print(f"\nRows: {df.shape[0]} | Columns: {df.shape[1]}")
print("\n✅ Dataset loaded successfully!")

In [ ]:
# Preview the first 5 rows
df.head()

In [ ]:
# Dataset Info
df.info()

In [ ]:
# Statistical Summary
df.describe()

---
## Step 3: Exploratory Data Analysis (EDA)

In [ ]:
# Check survival distribution
print("Survival Count:")
print(df['survived'].value_counts())
print(f"\nSurvival Rate: {df['survived'].mean()*100:.2f}%")

In [ ]:
# Check missing values
print("Missing Values per Column:")
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])

---
## Step 4: Data Visualization

In [ ]:
# ── Figure 1: Survival Count ──
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
survival_counts = df['survived'].value_counts()
axes[0].bar(['Did Not Survive', 'Survived'], survival_counts.values,
            color=['#e74c3c', '#2ecc71'], edgecolor='black', linewidth=0.8)
axes[0].set_title('Survival Count', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(survival_counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(survival_counts.values, labels=['Did Not Survive', 'Survived'],
            autopct='%1.1f%%', colors=['#e74c3c', '#2ecc71'],
            startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Survival Proportion', fontsize=14, fontweight='bold')

plt.suptitle('Overall Survival Distribution', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plot_survival_distribution.png', bbox_inches='tight')
plt.show()
print("📊 Figure 1 saved.")

In [ ]:
# ── Figure 2: Survival by Gender and Passenger Class ──
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Survival by Sex
sex_survival = df.groupby('sex')['survived'].mean() * 100
bars = axes[0].bar(sex_survival.index, sex_survival.values,
                   color=['#3498db', '#e91e63'], edgecolor='black', linewidth=0.8)
axes[0].set_title('Survival Rate by Gender', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Survival Rate (%)')
axes[0].set_ylim(0, 100)
for bar, val in zip(bars, sex_survival.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 1.5,
                 f'{val:.1f}%', ha='center', fontweight='bold')

# Survival by Pclass
pclass_survival = df.groupby('pclass')['survived'].mean() * 100
bars2 = axes[1].bar([f'Class {i}' for i in pclass_survival.index],
                    pclass_survival.values,
                    color=['#f39c12', '#27ae60', '#8e44ad'],
                    edgecolor='black', linewidth=0.8)
axes[1].set_title('Survival Rate by Passenger Class', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Survival Rate (%)')
axes[1].set_ylim(0, 100)
for bar, val in zip(bars2, pclass_survival.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 1.5,
                 f'{val:.1f}%', ha='center', fontweight='bold')

plt.suptitle('Survival Rate by Gender & Class', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_survival_gender_class.png', bbox_inches='tight')
plt.show()
print("📊 Figure 2 saved.")

In [ ]:
# ── Figure 3: Age Distribution ──
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Age histogram
df['age'].dropna().hist(bins=30, ax=axes[0], color='#3498db',
                         edgecolor='white', linewidth=0.5)
axes[0].set_title('Age Distribution (All Passengers)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Count')
axes[0].axvline(df['age'].median(), color='red', linestyle='--',
                linewidth=1.5, label=f'Median: {df["age"].median():.1f}')
axes[0].legend()

# Age by survived
df[df['survived'] == 0]['age'].dropna().hist(bins=25, ax=axes[1],
    alpha=0.6, color='#e74c3c', edgecolor='white', label='Did Not Survive')
df[df['survived'] == 1]['age'].dropna().hist(bins=25, ax=axes[1],
    alpha=0.6, color='#2ecc71', edgecolor='white', label='Survived')
axes[1].set_title('Age Distribution by Survival', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Age')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.suptitle('Age Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_age_distribution.png', bbox_inches='tight')
plt.show()
print("📊 Figure 3 saved.")

In [ ]:
# ── Figure 4: Fare Distribution ──
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Fare histogram
df['fare'].hist(bins=40, ax=axes[0], color='#9b59b6',
                edgecolor='white', linewidth=0.5)
axes[0].set_title('Fare Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Fare (£)')
axes[0].set_ylabel('Count')

# Fare by Pclass (Box plot)
df.boxplot(column='fare', by='pclass', ax=axes[1],
           patch_artist=True,
           boxprops=dict(facecolor='#3498db', color='navy'),
           medianprops=dict(color='red', linewidth=2))
axes[1].set_title('Fare by Passenger Class', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Passenger Class')
axes[1].set_ylabel('Fare (£)')
plt.suptitle('')  # Remove default boxplot title

plt.suptitle('Fare Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_fare_analysis.png', bbox_inches='tight')
plt.show()
print("📊 Figure 4 saved.")

In [ ]:
# ── Figure 5: Heatmap — Correlation Matrix ──
plt.figure(figsize=(10, 7))

numeric_df = df[['survived', 'pclass', 'age', 'sibsp', 'parch', 'fare']].copy()
corr_matrix = numeric_df.corr()

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    mask=mask,
    linewidths=0.5,
    linecolor='white',
    vmin=-1, vmax=1,
    square=True,
    cbar_kws={'shrink': 0.8}
)
plt.title('Feature Correlation Heatmap', fontsize=16, fontweight='bold', pad=15)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('plot_correlation_heatmap.png', bbox_inches='tight')
plt.show()
print("📊 Figure 5 saved.")

---
## Step 5: Data Cleaning & Preprocessing

In [ ]:
# Create a copy for preprocessing
df_clean = df.copy()

# ── Handle Missing Values ──
# Fill Age with median
median_age = df_clean['age'].median()
df_clean['age'].fillna(median_age, inplace=True)
print(f"✅ Age: filled {df['age'].isnull().sum()} missing values with median ({median_age:.1f})")

# Fill Fare with median (just in case)
df_clean['fare'].fillna(df_clean['fare'].median(), inplace=True)
print(f"✅ Fare: filled any missing values with median")

# Drop columns with too many missing values or not useful
cols_to_drop = ['deck', 'embark_town', 'alive', 'who', 'adult_male', 'alone', 'class', 'embarked']
df_clean.drop(columns=[c for c in cols_to_drop if c in df_clean.columns], inplace=True)
print(f"✅ Dropped unnecessary columns: {[c for c in cols_to_drop if c in df.columns]}")

print(f"\nRemaining columns: {list(df_clean.columns)}")
print(f"Missing values after cleaning: {df_clean.isnull().sum().sum()}")

---
## Step 6: Feature Engineering

In [ ]:
# ── Create FamilySize Feature ──
df_clean['FamilySize'] = df_clean['sibsp'] + df_clean['parch'] + 1  # +1 for self
print("✅ Created 'FamilySize' = sibsp + parch + 1")

# ── Encode 'sex' column (Male=1, Female=0) ──
le = LabelEncoder()
df_clean['sex_encoded'] = le.fit_transform(df_clean['sex'])
print("✅ Encoded 'sex': female=0, male=1")
print(f"   Classes: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# ── Final Feature Set ──
features = ['pclass', 'sex_encoded', 'age', 'fare', 'FamilySize']
target   = 'survived'

X = df_clean[features]
y = df_clean[target]

print(f"\n✅ Features selected: {features}")
print(f"   X shape: {X.shape}")
print(f"   y shape: {y.shape}")

X.head()

In [ ]:
# ── Figure 6: FamilySize vs Survival ──
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# FamilySize distribution
df_clean['FamilySize'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0], color='#1abc9c', edgecolor='black', linewidth=0.8
)
axes[0].set_title('FamilySize Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Family Size')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# FamilySize survival rate
fam_survival = df_clean.groupby('FamilySize')['survived'].mean() * 100
fam_survival.plot(kind='bar', ax=axes[1], color='#e67e22',
                  edgecolor='black', linewidth=0.8)
axes[1].set_title('Survival Rate by Family Size', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Family Size')
axes[1].set_ylabel('Survival Rate (%)')
axes[1].set_ylim(0, 100)
axes[1].axhline(y=df_clean['survived'].mean()*100, color='red',
                linestyle='--', linewidth=1.5, label='Overall Average')
axes[1].legend()
axes[1].tick_params(axis='x', rotation=0)

plt.suptitle('Family Size Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_familysize.png', bbox_inches='tight')
plt.show()
print("📊 Figure 6 saved.")

---
## Step 7: Model Building — Logistic Regression

In [ ]:
# ── Train/Test Split ──
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size : {X_train.shape[0]} samples")
print(f"Test set size     : {X_test.shape[0]} samples")
print(f"Train survival %  : {y_train.mean()*100:.1f}%")
print(f"Test survival %   : {y_test.mean()*100:.1f}%")

In [ ]:
# ── Feature Scaling ──
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
print("✅ Features scaled using StandardScaler")

In [ ]:
# ── Train Logistic Regression ──
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

print("✅ Logistic Regression model trained!")
print(f"   Solver: {model.solver}")
print(f"   Max iterations: {model.max_iter}")

---
## Step 8: Model Evaluation

In [ ]:
# ── Predictions ──
y_pred       = model.predict(X_test_scaled)
y_pred_train = model.predict(X_train_scaled)

# ── Accuracy Scores ──
train_acc = accuracy_score(y_train, y_pred_train) * 100
test_acc  = accuracy_score(y_test, y_pred)  * 100

print("=" * 40)
print("        MODEL PERFORMANCE")
print("=" * 40)
print(f"  Training Accuracy : {train_acc:.2f}%")
print(f"  Test Accuracy     : {test_acc:.2f}%")
print("=" * 40)

print("\nDetailed Classification Report (Test Set):")
print(classification_report(y_test, y_pred,
                             target_names=['Did Not Survive', 'Survived']))

In [ ]:
# ── Figure 7: Confusion Matrix ──
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion Matrix (raw counts)
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['Did Not Survive', 'Survived'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix\n(Counts)', fontsize=14, fontweight='bold')

# Confusion Matrix (normalized)
cm_norm = confusion_matrix(y_test, y_pred, normalize='true')
disp2 = ConfusionMatrixDisplay(confusion_matrix=cm_norm,
                                display_labels=['Did Not Survive', 'Survived'])
disp2.plot(ax=axes[1], colorbar=False, cmap='Greens')
axes[1].set_title('Confusion Matrix\n(Normalized)', fontsize=14, fontweight='bold')

plt.suptitle(f'Confusion Matrix — Test Accuracy: {test_acc:.2f}%',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_confusion_matrix.png', bbox_inches='tight')
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"\nTrue Negatives  (Correct: Did Not Survive): {tn}")
print(f"False Positives (Wrong:  Predicted Survived): {fp}")
print(f"False Negatives (Wrong:  Predicted Not Survived): {fn}")
print(f"True Positives  (Correct: Survived): {tp}")
print("\n📊 Figure 7 saved.")

In [ ]:
# ── Figure 8: Feature Importance (Logistic Regression Coefficients) ──
plt.figure(figsize=(9, 5))

coef_df = pd.DataFrame({
    'Feature': features,
    'Coefficient': model.coef_[0]
}).sort_values('Coefficient')

colors = ['#e74c3c' if c < 0 else '#2ecc71' for c in coef_df['Coefficient']]
bars = plt.barh(coef_df['Feature'], coef_df['Coefficient'],
                color=colors, edgecolor='black', linewidth=0.8)
plt.axvline(x=0, color='black', linewidth=1.2, linestyle='-')
plt.title('Feature Importance\n(Logistic Regression Coefficients)',
          fontsize=14, fontweight='bold')
plt.xlabel('Coefficient Value\n(Positive → increases survival probability)')

for bar, val in zip(bars, coef_df['Coefficient']):
    offset = 0.03 if val >= 0 else -0.03
    plt.text(val + offset, bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.savefig('plot_feature_importance.png', bbox_inches='tight')
plt.show()
print("📊 Figure 8 saved.")

---
## Step 9: Make Predictions on New Passengers

In [ ]:
# Predict survival for new passengers
# Features: [pclass, sex_encoded (0=female, 1=male), age, fare, FamilySize]

new_passengers = pd.DataFrame({
    'pclass':       [1,   3,   2,   1],
    'sex_encoded':  [0,   1,   0,   1],   # 0=female, 1=male
    'age':          [28,  22,  35,  45],
    'fare':         [100, 7.5, 20,  80],
    'FamilySize':   [2,   1,   3,   1]
})

new_scaled = scaler.transform(new_passengers)
predictions = model.predict(new_scaled)
probabilities = model.predict_proba(new_scaled)[:, 1]

new_passengers['Predicted'] = ['✅ SURVIVED' if p == 1 else '❌ NOT SURVIVED'
                                for p in predictions]
new_passengers['Survival Probability'] = [f"{p*100:.1f}%" for p in probabilities]
new_passengers['sex_encoded'] = new_passengers['sex_encoded'].map({0: 'Female', 1: 'Male'})

print("Prediction Results for New Passengers:")
print(new_passengers.to_string(index=False))

---
## Step 10: Summary & Conclusions

In [ ]:
print("="*55)
print("         PROJECT SUMMARY — TITANIC SURVIVAL")
print("="*55)
print(f"""
📦 Dataset:
   Total passengers  : {len(df)}
   Survival rate     : {df['survived'].mean()*100:.1f}%

🧹 Data Cleaning:
   Missing age values filled with median ({median_age:.1f} years)
   Unnecessary columns dropped

🔧 Feature Engineering:
   Created 'FamilySize' from sibsp + parch + 1
   Encoded 'sex' as binary numeric

📊 Visualizations Created:
   1. Survival Distribution (Count + Pie)
   2. Survival by Gender & Passenger Class
   3. Age Distribution (Overall + By Survival)
   4. Fare Analysis (Histogram + Boxplot)
   5. Correlation Heatmap
   6. Family Size Analysis
   7. Confusion Matrix (Counts + Normalized)
   8. Feature Importance (Coefficients)

🤖 Model: Logistic Regression
   Features          : {features}
   Training Accuracy : {train_acc:.2f}%
   Test Accuracy     : {test_acc:.2f}%

🔍 Key Insights:
   • Women had a much higher survival rate than men
   • 1st class passengers survived more than 3rd class
   • Small families (size 2–4) survived more often
   • Higher fare correlated with better survival chances
""")
print("="*55)
print("✅ Project Complete!")

---

## 📁 Saved Plots
All plots have been saved as PNG files in your working directory:

| File | Description |
|------|-------------|
| `plot_survival_distribution.png` | Overall survival count and proportion |
| `plot_survival_gender_class.png` | Survival rate by gender and class |
| `plot_age_distribution.png` | Age histograms |
| `plot_fare_analysis.png` | Fare distribution and boxplots |
| `plot_correlation_heatmap.png` | Feature correlation heatmap |
| `plot_familysize.png` | FamilySize distribution and survival |
| `plot_confusion_matrix.png` | Model confusion matrix |
| `plot_feature_importance.png` | Logistic regression coefficients |

---
*Project by: [Your Name] | Data Science | Titanic Dataset*